# AppleSupport Dev / Test Split

Builds `development_set.csv` and `test_set.csv` from `apple_support.csv`,
guaranteeing **zero overlap** with the 200 examples in
`golden_set_200_reviewed.csv`.

**Inputs required in the same folder as this notebook:**
- `apple_support.csv` — full raw AppleSupport Twitter dataset
- `golden_set_200_reviewed.csv` — your 200 human-reviewed golden evaluation examples

**Outputs produced:**
- `development_set.csv`
- `test_set.csv`

**Reproducibility:** the dev/test split uses `random_state=42`.

**Design decisions made explicit here (change the constants below if you want different behavior):**
- Only `inbound == True` rows (customer tweets) are kept — company replies from `AppleSupport` etc. are dropped.
- The dev/test split is 80% / 20%. This ratio wasn't specified in the original request, so it's a configurable constant, not a hidden default.
- Neither input file is modified. Both are only read.


In [ ]:
import pandas as pd
from pathlib import Path

# ---- Configuration ----
DATA_DIR = Path(".")                       # folder containing the two input CSVs
APPLE_SUPPORT_PATH = DATA_DIR / "apple_support.csv"
GOLDEN_SET_PATH = DATA_DIR / "golden_set_200_reviewed.csv"

OUTPUT_DIR = Path(".")
DEV_OUTPUT_PATH = OUTPUT_DIR / "development_set.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "test_set.csv"

RANDOM_STATE = 42
TEST_FRACTION = 0.20   # 80% development / 20% test

pd.set_option("display.max_colwidth", 120)

## Step 1 — Load and inspect `apple_support.csv`

Before doing anything else: how many rows, what columns exist, and can we
identify customer vs. company tweets from it?

In [ ]:
apple = pd.read_csv(APPLE_SUPPORT_PATH)

print("Rows:", len(apple))
print("Columns:", apple.columns.tolist())
print()
print(apple.dtypes)
print()
apple.head(3)

In [ ]:
# Can we identify customer/inbound tweets?
print(apple["inbound"].value_counts())
print()
print("Top author_ids for inbound=False (company replies):")
print(apple.loc[~apple["inbound"], "author_id"].value_counts().head(10))

In [ ]:
# Does apple_support.csv already contain any intent-taxonomy labels?
TAXONOMY_COLUMNS_EXPECTED = {
    "ACCOUNT_LOGIN", "APPLE_SERVICE_ISSUE", "PURCHASE_BILLING", "DEVICE_PERFORMANCE",
    "BATTERY_CHARGING", "IOS_UPDATE_ISSUE", "HARDWARE_REPAIR", "NETWORK_CONNECTIVITY",
    "FEATURE_SETTINGS", "CONTEXT_REQUIRED", "UNKNOWN_ESCALATE",
}
has_label_column = any(
    col.lower() in {"intent", "final_intent", "label", "ai_intent"}
    for col in apple.columns
)
print("apple_support.csv columns:", apple.columns.tolist())
print("Any intent/label-like column present?", has_label_column)
print()
if not has_label_column:
    print("CONCLUSION: apple_support.csv has NO intent labels of any kind.")
    print("A separate labeling/classification strategy is required before")
    print("training a TF-IDF baseline on development_set.csv / test_set.csv.")

## Step 2 — Load and inspect `golden_set_200_reviewed.csv`

This is the final, human-reviewed evaluation set. It must stay untouched,
and none of its `tweet_id`s may appear in the dev or test sets.

In [ ]:
golden = pd.read_csv(GOLDEN_SET_PATH)

print("Rows:", len(golden))
print("Columns:", golden.columns.tolist())
print("Unique tweet_ids:", golden["tweet_id"].nunique())
print("Duplicate tweet_ids:", golden["tweet_id"].duplicated().sum())
print()
golden.head(3)

In [ ]:
assert golden["tweet_id"].nunique() == len(golden), "golden set has duplicate tweet_ids!"
golden_ids = set(golden["tweet_id"])
print(f"Loaded {len(golden_ids)} unique golden tweet_ids.")

## Step 3 — Confirm golden tweet_ids exist in the raw dataset

Sanity check before removing anything: how many of the 200 golden
`tweet_id`s are actually present in `apple_support.csv`?

In [ ]:
golden_ids_found = golden_ids & set(apple["tweet_id"])
golden_ids_missing = golden_ids - set(apple["tweet_id"])

print(f"Golden tweet_ids found in apple_support.csv: {len(golden_ids_found)} / {len(golden_ids)}")
print(f"Golden tweet_ids missing from apple_support.csv: {len(golden_ids_missing)}")
if golden_ids_missing:
    print("Missing ids:", sorted(golden_ids_missing))

## Step 4 — Keep only customer (`inbound == True`) tweets, then remove golden ids

In [ ]:
customer = apple.loc[apple["inbound"] == True].copy()
customer_row_count = len(customer)

remaining = customer.loc[~customer["tweet_id"].isin(golden_ids)].copy()
removed_count = customer_row_count - len(remaining)

print(f"Customer/inbound rows: {customer_row_count:,}")
print(f"Rows removed for being in the golden set: {removed_count}")
print(f"Remaining customer rows (pre-split): {len(remaining):,}")

## Step 5 — Reproducible 80/20 development / test split

`random_state=42` makes this split deterministic and repeatable.

In [ ]:
test_set = remaining.sample(frac=TEST_FRACTION, random_state=RANDOM_STATE)
development_set = remaining.drop(test_set.index)

development_set = development_set.sort_values("tweet_id").reset_index(drop=True)
test_set = test_set.sort_values("tweet_id").reset_index(drop=True)

print(f"development_set: {len(development_set):,} rows")
print(f"test_set:        {len(test_set):,} rows")

## Step 6 — Verification checks

Every check below is computed from the actual dataframes (and, at the end,
from a fresh re-read of the files written to disk) — nothing here is
asserted without being tested.

In [ ]:
checks = {}

# no overlap between dev and test
checks["dev_test_no_overlap"] = len(set(development_set["tweet_id"]) & set(test_set["tweet_id"])) == 0

# zero overlap with golden set (the critical requirement)
checks["dev_vs_golden_overlap"] = len(set(development_set["tweet_id"]) & golden_ids)
checks["test_vs_golden_overlap"] = len(set(test_set["tweet_id"]) & golden_ids)

# uniqueness
checks["dev_unique_ids"] = development_set["tweet_id"].is_unique
checks["test_unique_ids"] = test_set["tweet_id"].is_unique

# every row really is a customer/inbound tweet
checks["dev_all_inbound"] = bool((development_set["inbound"] == True).all())
checks["test_all_inbound"] = bool((test_set["inbound"] == True).all())

# text wasn't modified from the source file
orig_text_map = dict(zip(apple["tweet_id"], apple["text"]))
dev_text_mismatch = [tid for tid, txt in zip(development_set["tweet_id"], development_set["text"])
                      if orig_text_map.get(tid) != txt]
test_text_mismatch = [tid for tid, txt in zip(test_set["tweet_id"], test_set["text"])
                       if orig_text_map.get(tid) != txt]
checks["dev_text_unmodified"] = len(dev_text_mismatch) == 0
checks["test_text_unmodified"] = len(test_text_mismatch) == 0

# conservation: dev + test + golden should equal total customer rows
checks["row_conservation"] = (len(development_set) + len(test_set) + len(golden)) == customer_row_count

for k, v in checks.items():
    print(f"{k:28s}: {v}")

In [ ]:
all_pass = all(bool(v) for k, v in checks.items()
               if k not in ("dev_vs_golden_overlap", "test_vs_golden_overlap"))
all_pass = all_pass and checks["dev_vs_golden_overlap"] == 0 and checks["test_vs_golden_overlap"] == 0

print("ALL CHECKS PASSED:", all_pass)
assert all_pass, "Validation failed -- see checks above before writing any files."


## Step 7 — Save outputs

Only `development_set.csv` and `test_set.csv` are written. Neither
`apple_support.csv` nor `golden_set_200_reviewed.csv` is ever opened in
write mode.

In [ ]:
development_set.to_csv(DEV_OUTPUT_PATH, index=False)
test_set.to_csv(TEST_OUTPUT_PATH, index=False)
print(f"Wrote {DEV_OUTPUT_PATH} ({len(development_set):,} rows)")
print(f"Wrote {TEST_OUTPUT_PATH} ({len(test_set):,} rows)")

## Step 8 — Independent re-verification

Re-read the files that were actually written to disk (not the in-memory
dataframes) and re-check the overlap and row-count guarantees from
scratch, plus confirm the two input files are unchanged.

In [ ]:
dev_check = pd.read_csv(DEV_OUTPUT_PATH)
test_check = pd.read_csv(TEST_OUTPUT_PATH)
apple_recheck = pd.read_csv(APPLE_SUPPORT_PATH)
golden_recheck = pd.read_csv(GOLDEN_SET_PATH)

print("development_set.csv reload -> rows:", len(dev_check), "| unique ids:", dev_check["tweet_id"].nunique())
print("test_set.csv reload        -> rows:", len(test_check), "| unique ids:", test_check["tweet_id"].nunique())
print()
print("Overlap dev vs golden:", len(set(dev_check["tweet_id"]) & golden_ids))
print("Overlap test vs golden:", len(set(test_check["tweet_id"]) & golden_ids))
print("Overlap dev vs test:", len(set(dev_check["tweet_id"]) & set(test_check["tweet_id"])))
print()
print("apple_support.csv unchanged (same shape on reload)?", apple_recheck.shape == apple.shape)
print("golden_set_200_reviewed.csv unchanged (same shape on reload)?", golden_recheck.shape == golden.shape)

## Summary report

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Original apple_support.csv rows",
        "Customer/inbound rows",
        "Golden set rows",
        "Golden tweet_ids found in original data",
        "Rows removed (golden overlap)",
        "Development set rows",
        "Test set rows",
        "Split ratio (dev/test)",
        "random_state",
        "Dev vs golden overlap",
        "Test vs golden overlap",
        "Dev vs test overlap",
    ],
    "value": [
        len(apple),
        customer_row_count,
        len(golden),
        f"{len(golden_ids_found)} / {len(golden_ids)}",
        removed_count,
        len(development_set),
        len(test_set),
        f"{int((1-TEST_FRACTION)*100)}% / {int(TEST_FRACTION*100)}%",
        RANDOM_STATE,
        len(set(dev_check['tweet_id']) & golden_ids),
        len(set(test_check['tweet_id']) & golden_ids),
        len(set(dev_check['tweet_id']) & set(test_check['tweet_id'])),
    ],
})
summary